# 02 Split and features

We split students into train / validation / test (70 / 15 / 15), so the test set is made of new students. Then we build the features from earlier answers only.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from src.data import load_data, split_students

df = split_students(load_data())
df.shape
df.head()

,QuestionId,UserId,AnswerId,IsCorrect,CorrectAnswer,AnswerValue,DateAnswered,Confidence,GroupId,QuizId,SchemeOfWorkId,Gender,DateOfBirth,PremiumPupil,SubjectId,skill_L0,skill_L1,skill_L2,skill_L3,split
0,182,1,1181805,0,3,2,2019-10-11 13:15:00,NaN,251,75,52562.0,0,NaN,NaN,"[3, 71, 98, 209]",3,71,98,209,train
1,202,1,287821,0,1,2,2019-10-11 13:17:00,50.0,251,75,52562.0,0,NaN,NaN,"[3, 71, 98, 209]",3,71,98,209,train
2,420,1,925509,1,2,2,2019-10-11 13:17:00,50.0,251,75,52562.0,0,NaN,NaN,"[3, 71, 98, 209]",3,71,98,209,train
3,403,1,1192500,1,3,3,2019-10-11 13:18:00,50.0,251,75,52562.0,0,NaN,NaN,"[3, 71, 98, 209]",3,71,98,209,train
4,691,1,522876,1,3,3,2019-10-11 13:19:00,50.0,251,75,52562.0,0,NaN,NaN,"[3, 71, 98, 209]",3,71,98,209,train


## 1. Check the split

In [8]:
summary = pd.DataFrame({
    "students": df.groupby("split")["UserId"].nunique(),
    "answers": df["split"].value_counts(),
    "% correct": (df.groupby("split")["IsCorrect"].mean() * 100).round(1),
    "% free school meals": (df.groupby("split")["PremiumPupil"].mean() * 100).round(1),
}).reindex(["train", "val", "test"]).rename_axis("split").reset_index()

print("Students in more than one split:", (df.groupby("UserId")["split"].nunique() > 1).sum())
summary

Students in more than one split: 0


,split,students,answers,% correct,% free school meals
0,train,3442,977349,53.6,28.0
1,val,737,200546,54.7,24.9
2,test,739,204832,53.2,30.3


**Takeaway:** 3,442 / 737 / 739 students (70.7% / 14.5% / 14.8% of answers). No student appears in more than one split, so the test set is made of new students. The correct rate is similar in all splits (53.2–54.7%). The free-school-meal share varies a little (24.9–30.3% of known students) by chance, which we note as a limitation for the group comparison.

## 2. Features

For every answer we add clues from the student's past and about the question.
Clues only use **answers from earlier minutes** by the same student, or **training students only**, so the model never sees the answer it has to predict.

| Feature | Meaning |
|---|---|
| `n_prev` | how many questions the student answered before |
| `acc_prev` | the student's accuracy so far |
| `n_prev_L1` … `L3`, `acc_prev_L1` … `L3` | the same, but only on this question's topic (level 1, 2 or 3) |
| `q_correct_rate` | share of training students who got this question right |

Accuracy is smoothed as (correct + 1) / (answers + 2), so a student with no history starts at 0.5.

In [ ]:
new_cols = [c for c in df.columns if c.startswith(("n_prev", "correct_prev", "acc_prev")) or c == "q_correct_rate"]
print("Rows, columns:", df.shape)
print("New columns:", len(new_cols))

meaning = {
    "n_prev":          "answers before (all questions)",
    "correct_prev":    "right answers before (all questions)",
    "acc_prev":        "accuracy so far (all questions)",
    "n_prev_L0":       "answers before (level 0 topic = Maths)",
    "correct_prev_L0": "right answers before (level 0 topic)",
    "acc_prev_L0":     "accuracy so far (level 0 topic)",
    "n_prev_L1":       "answers before (level 1 topic)",
    "correct_prev_L1": "right answers before (level 1 topic)",
    "acc_prev_L1":     "accuracy so far (level 1 topic)",
    "n_prev_L2":       "answers before (level 2 topic)",
    "correct_prev_L2": "right answers before (level 2 topic)",
    "acc_prev_L2":     "accuracy so far (level 2 topic)",
    "n_prev_L3":       "answers before (level 3 topic)",
    "correct_prev_L3": "right answers before (level 3 topic)",
    "acc_prev_L3":     "accuracy so far (level 3 topic)",
    "q_correct_rate":  "share of training students who got this question right",
}

new_features = pd.DataFrame({
    "meaning": meaning,
    "min": df[new_cols].min(),
    "median": df[new_cols].median(),
    "mean": df[new_cols].mean(),
    "max": df[new_cols].max(),
    "missing": df[new_cols].isna().sum(),
}).round(2)

new_features

Rows, columns: (1382727, 36)
New columns: 16
['n_prev', 'correct_prev', 'acc_prev', 'n_prev_L0', 'correct_prev_L0', 'acc_prev_L0', 'n_prev_L1', 'correct_prev_L1', 'acc_prev_L1', 'n_prev_L2', 'correct_prev_L2', 'acc_prev_L2', 'n_prev_L3', 'correct_prev_L3', 'acc_prev_L3', 'q_correct_rate']
Rows, columns: (1382727, 36)
New columns: 16


,UserId,QuestionId,DateAnswered,split,IsCorrect,n_prev,correct_prev,acc_prev,n_prev_L0,correct_prev_L0,acc_prev_L0,n_prev_L1,correct_prev_L1,acc_prev_L1,n_prev_L2,correct_prev_L2,acc_prev_L2,n_prev_L3,correct_prev_L3,acc_prev_L3,q_correct_rate
0,1,182,2019-10-11 13:15:00,train,0,0,0,0.500000,0,0,0.500000,0,0,0.500000,0,0,0.500000,0,0,0.500000,0.267171
1,1,202,2019-10-11 13:17:00,train,0,1,0,0.333333,1,0,0.333333,1,0,0.333333,1,0,0.333333,1,0,0.333333,0.778200
2,1,420,2019-10-11 13:17:00,train,1,1,0,0.333333,1,0,0.333333,1,0,0.333333,1,0,0.333333,1,0,0.333333,0.769133
3,1,403,2019-10-11 13:18:00,train,1,3,1,0.400000,3,1,0.400000,3,1,0.400000,3,1,0.400000,3,1,0.400000,0.652537
4,1,691,2019-10-11 13:19:00,train,1,4,2,0.500000,4,2,0.500000,4,2,0.500000,4,2,0.500000,4,2,0.500000,0.661499


**Takeaway:** We added 16 new columns, so the table now has 36. Every answer now has its clues next to it.

### Check by hand: the first answers of one student
Answers in the same minute should get the same history.

In [10]:
first = df[df["UserId"] == df["UserId"].iloc[0]]
first[["DateAnswered", "QuestionId", "IsCorrect", "n_prev", "correct_prev", "acc_prev"]].head(10)

,DateAnswered,QuestionId,IsCorrect,n_prev,correct_prev,acc_prev
0,2019-10-11 13:15:00,182,0,0,0,0.500000
1,2019-10-11 13:17:00,202,0,1,0,0.333333
2,2019-10-11 13:17:00,420,1,1,0,0.333333
3,2019-10-11 13:18:00,403,1,3,1,0.400000
4,2019-10-11 13:19:00,691,1,4,2,0.500000
5,2019-10-11 13:20:00,261,0,5,3,0.571429
6,2019-10-11 13:20:00,503,0,5,3,0.571429
7,2019-10-11 13:21:00,289,0,7,3,0.444444
8,2019-10-11 13:21:00,566,1,7,3,0.444444
9,2019-10-11 13:22:00,241,1,9,4,0.454545


**Takeaway:** The two answers at 13:17 both have 1 earlier answer, so they don't count each other. The history only grows when the minute changes. The "earlier minutes only" rule works.

### How much history do students have?

In [11]:
history = pd.DataFrame({
    "level": ["all questions", "level 1 topic", "level 2 topic", "level 3 topic"],
    "earlier answers (median)": [df["n_prev"].median(), df["n_prev_L1"].median(),
                                 df["n_prev_L2"].median(), df["n_prev_L3"].median()],
    "accuracy so far (median)": [df["acc_prev"].median(), df["acc_prev_L1"].median(),
                                 df["acc_prev_L2"].median(), df["acc_prev_L3"].median()],
}).round(2)

print("Missing values in the features:", df[new_cols].isna().sum().sum())
history

Missing values in the features: 0


,level,earlier answers (median),accuracy so far (median)
0,all questions,160.0,0.56
1,level 1 topic,62.0,0.55
2,level 2 topic,15.0,0.53
3,level 3 topic,6.0,0.50


**Takeaway:** No feature has missing values. A typical answer has 160 earlier answers overall, but only 62 on its level-1 topic, 15 on its level-2 topic and 6 on its level-3 topic. The more specific the topic, the less history there is.

### Level 0 is the same as "all questions"
Every question's level-0 topic is "Maths", so level-0 history should be the same as overall history.

In [12]:
print("acc_prev_L0 same as acc_prev:", (df["acc_prev_L0"] == df["acc_prev"]).all())
print("n_prev_L0 same as n_prev:    ", (df["n_prev_L0"] == df["n_prev"]).all())

acc_prev_L0 same as acc_prev: True
n_prev_L0 same as n_prev:     True


**Takeaway:** They are exactly the same, so the models only use `acc_prev` and `n_prev`. Using both copies would confuse logistic regression.

### Leakage check
We score each clue on its own, on training and on validation students. If a clue secretly contained the answer, it would look much better on training.

In [13]:
from sklearn.metrics import roc_auc_score

clues = {
    "accuracy so far (all questions)": "acc_prev",
    "accuracy on level 1 topic": "acc_prev_L1",
    "accuracy on level 2 topic": "acc_prev_L2",
    "accuracy on level 3 topic": "acc_prev_L3",
    "question difficulty": "q_correct_rate",
}

rows = []
for name, col in clues.items():
    rows.append({
        "clue": name,
        "AUC train": round(roc_auc_score(df.loc[df["split"] == "train", "IsCorrect"], df.loc[df["split"] == "train", col]), 3),
        "AUC val": round(roc_auc_score(df.loc[df["split"] == "val", "IsCorrect"], df.loc[df["split"] == "val", col]), 3),
    })

pd.DataFrame(rows)

,clue,AUC train,AUC val
0,accuracy so far (all questions),0.701,0.701
1,accuracy on level 1 topic,0.699,0.700
2,accuracy on level 2 topic,0.683,0.683
3,accuracy on level 3 topic,0.665,0.665
4,question difficulty,0.673,0.675


**Takeaway:** Every clue scores the same on training and validation students, so no clue is cheating.
On its own, overall accuracy is the best clue (AUC 0.70). The narrower the topic, the weaker the clue (level 3: 0.67), because there is less history. Question difficulty alone gets 0.68, and our models must beat this.
In notebook 03 we test if topic clues help *together* with overall accuracy.